# Phase 3 — Train Qwen bằng privileged distillation

Notebook này chỉ chuẩn bị Kaggle, tìm output notebook 01/02 và chạy `train_model.py`. Data loading, ba serving mode, teacher/student forward, CE+KL, evaluation và checkpoint đều nằm trong source; hyperparameter nằm trong Gin.

## Trước khi chạy

1. Bật GPU T4 trong Kaggle Notebook Settings.
2. Add Data chứa output notebook 01 và notebook 02.
3. Tạo Kaggle Secrets `GITHUB_TOKEN` và `WANDB_API_KEY`.

In [ ]:
from pathlib import Path

PREPROCESSED_ROOT = None
SID_ROOT = None

GITHUB_REPOSITORY_URL = "https://github.com/nam-htran/VSF-MiniApp-Ecommerce.git"
GITHUB_BRANCH = "main"
REPOSITORY_ROOT = Path("/kaggle/working/vsf-miniapp-ecommerce-source")
OUTPUT_ROOT = Path("/kaggle/working/model")

print("Configuration loaded.")

## 1. Cài dependency và kiểm tra GPU

In [ ]:
import os
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "accelerate>=1.0.0", "gin-config==0.5.0", "pyarrow>=16.0.0",
    "polars>=1.39.0", "transformers>=4.51.0,<5.0.0", "wandb>=0.19.0",
    "bitsandbytes>=0.43.0",
])

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")
print("GPU:", torch.cuda.get_device_name(0))

## 2. Kết nối Weights & Biases

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## 3. Clone source từ GitHub

In [ ]:
github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPOSITORY_ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()
git_environment = {
    **os.environ,
    "GITHUB_TOKEN": github_token,
    "GIT_TERMINAL_PROMPT": "0",
}
credential_helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
git = ["git", "-c", f"credential.helper={credential_helper}"]

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(
        [*git, "-C", str(REPOSITORY_ROOT), "pull", "--ff-only", "origin", GITHUB_BRANCH],
        check=True, env=git_environment,
    )
elif REPOSITORY_ROOT.exists():
    raise FileExistsError(f"Clone target is not a Git repository: {REPOSITORY_ROOT}")
else:
    subprocess.run(
        [*git, "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True, env=git_environment,
    )
del github_token, git_environment

SOURCE_ROOT = REPOSITORY_ROOT / "pi-recommendation/src"
if not (SOURCE_ROOT / "train_model.py").is_file():
    raise FileNotFoundError(f"Phase 3 source not found: {SOURCE_ROOT}")
print("SOURCE_ROOT:", SOURCE_ROOT)

## 4. Tìm artifacts và tạo Gin runtime

In [ ]:
import json


def locate_root(explicit, required_file):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if (root / required_file).is_file():
            return root
        raise FileNotFoundError(f"{required_file} was not found in {root}")

    candidates = [
        Path("/kaggle/working/preprocessed"),
        Path("/kaggle/working/rq-vae"),
    ]
    candidates.extend(path.parent for path in Path("/kaggle/input").glob(f"**/{required_file}"))
    for root in candidates:
        if (root / required_file).is_file():
            return root.resolve()
    raise FileNotFoundError(f"Could not locate {required_file}")


PREPROCESSED_ROOT = locate_root(PREPROCESSED_ROOT, "ranking.parquet")
SID_ROOT = locate_root(SID_ROOT, "semantic_ids.parquet")
SEMANTIC_IDS_PATH = SID_ROOT / "semantic_ids.parquet"

BASE_CONFIG_PATH = SOURCE_ROOT / "configs/model_kuaisearch.gin"
CONFIG_PATH = Path("/kaggle/working/model_kuaisearch.gin")
config_lines = BASE_CONFIG_PATH.read_text(encoding="utf-8").splitlines()
bindings = {
    "train.preprocessed_folder=": str(PREPROCESSED_ROOT),
    "train.semantic_ids_path=": str(SEMANTIC_IDS_PATH),
    "train.save_dir_root=": str(OUTPUT_ROOT),
}
for binding, value in bindings.items():
    matches = [index for index, line in enumerate(config_lines) if line.startswith(binding)]
    if len(matches) != 1:
        raise ValueError(f"Expected one {binding} binding, found {len(matches)}")
    config_lines[matches[0]] = binding + json.dumps(value)
CONFIG_PATH.write_text("\n".join(config_lines) + "\n", encoding="utf-8")

print("PREPROCESSED_ROOT:", PREPROCESSED_ROOT)
print("SEMANTIC_IDS_PATH:", SEMANTIC_IDS_PATH)
print("Gin config:", CONFIG_PATH)

## 5. Train model

In [ ]:
command = [sys.executable, "train_model.py", str(CONFIG_PATH)]
print("Running:", " ".join(command))
subprocess.run(command, cwd=SOURCE_ROOT, check=True)

## 6. Kiểm tra artifacts

In [ ]:
CHECKPOINT_PATH = OUTPUT_ROOT / "best_checkpoint.pt"
TOKENIZER_ROOT = OUTPUT_ROOT / "tokenizer"
CONFIG_OUTPUT_PATH = OUTPUT_ROOT / "training_config.json"

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError("Training finished without best_checkpoint.pt")
if not (TOKENIZER_ROOT / "tokenizer_config.json").is_file():
    raise FileNotFoundError("Training finished without tokenizer artifacts")
if not CONFIG_OUTPUT_PATH.is_file():
    raise FileNotFoundError("Training finished without training_config.json")

print("Artifact validation: PASSED")
print("Checkpoint:", CHECKPOINT_PATH)
print("Tokenizer:", TOKENIZER_ROOT)
print(CONFIG_OUTPUT_PATH.read_text(encoding="utf-8"))
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Notebook hoàn thành khi cell cuối báo `Artifact validation: PASSED`.